# Matching semántico por embeddings — versión mejorada

**Proyecto BME513 · prueba definitiva del refuerzo del módulo 2**

La primera versión (modelo multilingüe genérico) rindió por debajo de las reglas:
concordancia 69% y 3/8 en frases atípicas, por dos causas: (1) desconocimiento de
vocabulario clínico y (2) confusión entre categorías de imagen que se solapan.

Esta versión ataca ambas causas para dar al enfoque su **mejor oportunidad** antes de
descartarlo:
1. **Modelos clínicos en español** (biomédicos), no genéricos.
2. **Frases ancla ampliadas y afinadas** por categoría.
3. **Comparación de varios modelos** para elegir el mejor candidato.

Si aun con el mejor modelo clínico el rendimiento sigue por debajo de las reglas, el
descarte queda respaldado con doble evidencia.


## 1. Instalación

In [1]:
!pip install -q -U sentence-transformers
import numpy as np, pandas as pd, torch
from sentence_transformers import SentenceTransformer, models
print("torch:", torch.__version__, "| GPU:", torch.cuda.is_available())


torch: 2.11.0+cu128 | GPU: True


/tmp/ipykernel_442/2960277600.py:3: DeprecationWarning: Importing from 'sentence_transformers.models' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.modules' instead.
  from sentence_transformers import SentenceTransformer, models


## 2. Cargar modelos a comparar

Se prueban un modelo clínico en español (con *mean-pooling* sobre el encoder) y modelos
de oraciones. El comparativo elige el mejor.

In [2]:
def cargar_como_sentence_model(nombre_hf):
    """Envuelve un encoder HF (p.ej. bsc-bio-ehr-es) como SentenceTransformer con mean-pooling."""
    word = models.Transformer(nombre_hf, max_seq_length=128)
    pool = models.Pooling(word.get_word_embedding_dimension(), pooling_mode="mean")
    return SentenceTransformer(modules=[word, pool])

MODELOS = {}
# Clínico en español (biomédico + clínico) -- el candidato fuerte
try:
    MODELOS["bsc-bio-ehr-es (clínico)"] = cargar_como_sentence_model("PlanTL-GOB-ES/bsc-bio-ehr-es")
    print("OK: bsc-bio-ehr-es")
except Exception as e:
    print("No se pudo cargar bsc-bio-ehr-es:", e)
# RoBERTa biomédico español
try:
    MODELOS["roberta-biomedical-es"] = cargar_como_sentence_model("PlanTL-GOB-ES/roberta-base-biomedical-clinical-es")
    print("OK: roberta-biomedical-clinical-es")
except Exception as e:
    print("No se pudo cargar roberta biomedical:", e)
# Sentence-model español dedicado (referencia)
try:
    MODELOS["sentence-es (referencia)"] = SentenceTransformer("hiiamsid/sentence_similarity_spanish_es")
    print("OK: sentence_similarity_spanish_es")
except Exception as e:
    print("No se pudo cargar sentence-es:", e)

print("\nModelos disponibles:", list(MODELOS.keys()))


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/613 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: PlanTL-GOB-ES/bsc-bio-ehr-es
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.decoder.bias      | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


...

vocab.txt:   0%|          | 0.00/242k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/480k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

OK: sentence_similarity_spanish_es

Modelos disponibles: ['bsc-bio-ehr-es (clínico)', 'roberta-biomedical-es', 'sentence-es (referencia)']


## 3. Categorías con frases ancla ampliadas y afinadas

Se amplían las anclas, incluyendo sinónimos clínicos y coloquiales, y se refuerza la
distinción entre las categorías de imagen que antes se confundían.

In [3]:
CATEGORIAS = {
    "biopsia_histologia": [
        "se recomienda biopsia", "punción con aguja gruesa para histología",
        "estudio percutáneo con aguja", "toma de muestra tisular",
        "confirmación histológica de la lesión", "core biopsy con marcación",
        "punción aspirativa con aguja fina", "estudio anatomopatológico",
    ],
    "derivacion_oncologica": [
        "derivación a oncología", "referir a mastólogo", "valoración por especialista de mama",
        "derivar a comité oncológico", "interconsulta con cirujano de mama",
    ],
    "estudio_complementario_imagen": [
        "estudio complementario de imagen", "proyecciones adicionales focalizadas",
        "incidencias con magnificación", "compresión localizada",
        "completar el estudio mamográfico con más proyecciones",
    ],
    "correlacion_ecografica": [
        "correlación con ecografía mamaria", "complementar con ultrasonido mamario",
        "ecografía dirigida de la mama", "sonografía mamaria complementaria",
        "ecografía por patrón mamográfico denso",
    ],
    "comparacion_estudios_previos": [
        "comparar con estudios previos", "correlacionar con mamografías anteriores",
        "cotejar con exámenes anteriores del archivo", "revisar controles previos",
    ],
    "control_corto_plazo": [
        "control a corto plazo en 6 meses", "repetir mamografía en medio año",
        "seguimiento precoz en tres meses", "control mamográfico semestral",
    ],
    "control_anual": [
        "control mamográfico anual de rutina", "seguimiento habitual en doce meses",
        "control en un año", "screening anual de rutina",
    ],
    "criterio_medico": [
        "según criterio del médico tratante", "a consideración del clínico",
        "conducta a definir por el médico",
    ],
}
nombres_cat = list(CATEGORIAS.keys())
print("Categorías:", len(nombres_cat), "| anclas totales:", sum(len(v) for v in CATEGORIAS.values()))


Categorías: 8 | anclas totales: 38


## 4. Función de evaluación (concordancia + frases atípicas) por modelo

In [4]:
CASOS_ATIPICOS = [
    ("se sugiere estudio percutáneo con aguja gruesa", "biopsia_histologia"),
    ("amerita valoración por mastólogo", "derivacion_oncologica"),
    ("conviene complementar con sonografía mamaria", "correlacion_ecografica"),
    ("repetir mamografía en medio año", "control_corto_plazo"),
    ("seguimiento imagenológico dentro de un año", "control_anual"),
    ("procede toma de muestra tisular", "biopsia_histologia"),
    ("cotejar con exámenes anteriores del archivo", "comparacion_estudios_previos"),
    ("proyecciones adicionales focalizadas", "estudio_complementario_imagen"),
]

def preparar(modelo):
    matriz = []
    for cat in nombres_cat:
        emb = modelo.encode(CATEGORIAS[cat], normalize_embeddings=True)
        matriz.append(np.mean(emb, axis=0))
    return np.stack(matriz)

def clasificar_lote(modelo, matriz, textos, batch=64):
    emb = modelo.encode(list(textos), normalize_embeddings=True, batch_size=batch)
    sims = emb @ matriz.T
    idx = sims.argmax(axis=1)
    return [nombres_cat[i] for i in idx], sims.max(axis=1)

def evaluar(modelo, ref):
    matriz = preparar(modelo)
    pred, _ = clasificar_lote(modelo, matriz, ref["texto"].astype(str).tolist())
    concord = np.mean(np.array(pred) == ref["categoria_reglas"].values)
    at_pred, _ = clasificar_lote(modelo, matriz, [f for f,_ in CASOS_ATIPICOS])
    aciertos = sum(p==e for p,(_,e) in zip(at_pred, CASOS_ATIPICOS))
    return concord, aciertos, pred, at_pred


## 5. Cargar el corpus de referencia y comparar modelos

In [5]:
from google.colab import files
import io
print("Sube recomendaciones_clasificadas.csv ...")
sub = files.upload()
ref = pd.read_csv(io.BytesIO(list(sub.values())[0]))
print(f"{len(ref)} recomendaciones cargadas.\n")

resultados = {}
for nombre, modelo in MODELOS.items():
    print(f"Evaluando: {nombre} ...")
    concord, aciertos, pred, at_pred = evaluar(modelo, ref)
    resultados[nombre] = {"concordancia": concord, "atipicas": aciertos,
                          "pred": pred, "at_pred": at_pred}
    print(f"   concordancia={100*concord:.1f}%  |  atípicas={aciertos}/8\n")

print("="*55)
print(f"{'Modelo':<28}{'Concord.':>10}{'Atípicas':>12}")
print("-"*55)
print(f"{'REGLAS (baseline)':<28}{'100.0%':>10}{'8/8*':>12}")
for n, r in sorted(resultados.items(), key=lambda x:-x[1]["concordancia"]):
    print(f"{n:<28}{100*r['concordancia']:>9.1f}%{str(r['atipicas'])+'/8':>12}")
print("="*55)
print("* las reglas se toman como referencia; las atípicas son el caso donde el embedding debería superarlas")


Sube recomendaciones_clasificadas.csv ...


Saving recomendaciones_clasificadas.csv to recomendaciones_clasificadas.csv
4349 recomendaciones cargadas.

Evaluando: bsc-bio-ehr-es (clínico) ...
   concordancia=77.4%  |  atípicas=7/8

Evaluando: roberta-biomedical-es ...
   concordancia=57.5%  |  atípicas=8/8

Evaluando: sentence-es (referencia) ...
   concordancia=59.2%  |  atípicas=8/8

Modelo                        Concord.    Atípicas
-------------------------------------------------------
REGLAS (baseline)               100.0%        8/8*
bsc-bio-ehr-es (clínico)         77.4%         7/8
sentence-es (referencia)         59.2%         8/8
roberta-biomedical-es            57.5%         8/8
* las reglas se toman como referencia; las atípicas son el caso donde el embedding debería superarlas


## 6. Diagnóstico del mejor modelo: ¿dónde sigue fallando?

In [6]:
mejor = max(resultados, key=lambda k: resultados[k]["concordancia"])
print(f"Mejor modelo: {mejor}  ({100*resultados[mejor]['concordancia']:.1f}% concordancia)\n")
ref["cat_emb"] = resultados[mejor]["pred"]
desac = ref[ref["cat_emb"] != ref["categoria_reglas"]]
print(f"Desacuerdos con reglas: {len(desac)} ({100*len(desac)/len(ref):.1f}%)\n")
if len(desac):
    print("Confusiones más frecuentes (reglas -> embedding):")
    conf = desac.groupby(["categoria_reglas","cat_emb"]).size().sort_values(ascending=False)
    print(conf.head(10).to_string())

print("\nFrases atípicas con el mejor modelo:")
for (frase, esp), pred in zip(CASOS_ATIPICOS, resultados[mejor]["at_pred"]):
    print(f"  {'✓' if pred==esp else '✗'} '{frase}' -> {pred} (esp {esp})")


Mejor modelo: bsc-bio-ehr-es (clínico)  (77.4% concordancia)

Desacuerdos con reglas: 982 (22.6%)

Confusiones más frecuentes (reglas -> embedding):
categoria_reglas               cat_emb               
estudio_complementario_imagen  correlacion_ecografica    828
biopsia_histologia             correlacion_ecografica     53
control_corto_plazo            control_anual              27
criterio_medico                control_anual              24
control_anual                  correlacion_ecografica     18
criterio_medico                correlacion_ecografica     10
biopsia_histologia             criterio_medico             6
comparacion_estudios_previos   correlacion_ecografica      5
criterio_medico                control_corto_plazo         3
control_anual                  control_corto_plazo         2

Frases atípicas con el mejor modelo:
  ✓ 'se sugiere estudio percutáneo con aguja gruesa' -> biopsia_histologia (esp biopsia_histologia)
  ✓ 'amerita valoración por mastólogo' -> derivac

## 7. Veredicto

Compara el mejor modelo clínico contra las reglas:

- **Si concordancia ≥90% y atípicas ≥7/8** → el enfoque clínico sí aporta; reconsiderar
  integrarlo como *fallback* del módulo 2.
- **Si sigue por debajo** → queda descartado con doble evidencia (genérico y clínico). La
  causa estructural es que las tres categorías de imagen (estudio complementario /
  correlación ecográfica / comparación previos) son semánticamente cercanas, y un
  clasificador por significado tiende a fundirlas, mientras que las reglas las separan
  por señales explícitas. Se documenta como experimento negativo que respalda la decisión
  de diseño basada en reglas.
